# CS 431: Data-Intensive Distributed Computing (Fall 2026)
# Assignment 5: Inverted Indexing and Boolean Retrieval

So far, you've counted words in product reviews (Assignment 1), built and analyzed a data warehouse from the [Steam platform](https://store.steampowered.com/) (Assignments 2 and 3), and trained machine learning models (Assignment 4).
This assignment is about unstructured text and the data structure at the heart of every search engine: the _inverted index_.

For this assignment, you'll be using a sample of English Wikipedia comprising about 10.8 million sentences.
You'll build:

+ An indexer in PySpark that creates a plain-text inverted index.
+ A simple Boolean search engine that answers queries using the plain-text index.
+ An indexer in PySpark that creates a compressed (binary-encoded) inverted index, using gap encoding and variable-byte codes.
+ The same Boolean search engine, running over the compressed index.

Along the way, you'll measure what compression buys you and what it costs.

You're expected to use a coding agent to help you with this assignment.
However, you are ultimately responsible for and accountable for _everything_, including design decisions, quality checks, bugs, edge cases, interpretation of the results, etc.
You are also responsible for _understanding_ what the coding agent is doing.

<font color="red"><b>Important Consideration</b></font>:
This assignment asks a number of short-answer questions.
Answer completely but succinctly.
We don't expect an overly verbose explanation: simply enough to convey your understanding of the material.
LLMs often use a lot of words but convey little actual content.

Your coding agent will likely be able to one-shot the answer, but you should resist that temptation because it will likely impede your understanding if you simply ask the LLM to give you the answer.
As a suggestion, avoid simply copying/pasting the question into the prompt or directly pointing your agent at a specific notebook cell.
Try to come up with the answer yourself, perhaps using the agent to explain related concepts.
In other words, use the agent to help you _understand_ the material, not write the answer for you.
Such a strategy might be more conducive to learning.

This assignment has two kinds of tasks:

+ In many places, we give you working code and ask you to explain it. Your answers should be about _this_ code and _your_ outputs (the numbers, files, and timings you see), not generic textbook descriptions.
+ In other places, you implement core functions to a precise specification. The checks we provide are not exhaustive: when grading, we'll also call your functions on inputs you haven't seen. So implement the specification, not just enough to pass the checks.

For each short answer, two or three clear sentences are usually enough. Refer to the displayed evidence rather than giving a general textbook explanation. Use your own measured values where requested; timings may differ between runs.

When doing this assignment:

+ Use PySpark RDDs (not DataFrames) to process the corpus and to build the indexes. Do not read the corpus into Python data structures on the driver, except where provided code does so.
+ With 10,805,214 documents, it might be unwieldy to work with the entire corpus all at once, especially while debugging. Feel free to try your code on a subset first (e.g., `docs.filter(lambda doc: doc[0] < 100_000)`), but your submitted notebook must run on the full corpus.
+ Use exactly the names, signatures, keys, and output formats that we specify.
+ Retrieval (Sections 3 and 5) runs in plain Python in this notebook's process, using the functions you write.
+ Keep the provided cells unchanged unless a `TODO` tells you otherwise.

To start, let's set up Spark. Run the following code and resolve any errors before continuing.

Plan for approximately **24 GB of RAM** for the complete notebook, with 32 GB recommended, and about **10 GB of free disk space** for the corpus and three index variants. 
These are planning estimates, not measured peak requirements for every machine.

Spark's `spark.driver.memory` option configures the JVM heap; it does not cap the notebook's Python process or the aggregate memory of Python workers.
The higher driver memory and settings support the collection step in Section 4.3.

Use the course Python environment with PySpark, pandas, and a Jupyter kernel installed.

In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable

# TODO: Set SPARK_HOME for your machine if PySpark is not already installed.
# os.environ["SPARK_HOME"] = "/path/to/spark-4.2.0-bin-hadoop3"
if os.getenv("SPARK_HOME"):
    import findspark
    findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Assignment5").master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("spark_version=" + spark.version)

## 1. The Corpus

### 1.1 Get the Corpus

We'll be working with `corpus-enwiki-sample.txt.gz` (716 MB), from `https://cs.uwaterloo.ca/~jimmylin/cs431-2026f/corpus-enwiki-sample.txt.gz`.
The code cell below takes care of getting it for you: it downloads the file into a `data/` directory next to this notebook, verifies its MD5 checksum, and uncompresses it (1.8 GB).
Everything in this assignment uses the uncompressed file.

The corpus is a sample of sentences from Wikipedia.
For this assignment, we'll be treating each sentence from Wikipedia as a separate "document", so in fact we'll be indexing and searching Wikipedia sentences.

Terminological note: the "corpus", the "(document) collection", and other related variants are used interchangeably to refer to "the pile of text" we're indexing and searching over.

Run the cell below.

In [ ]:
import gzip
import hashlib
import os
import shutil
import urllib.request

CORPUS_URL = "https://cs.uwaterloo.ca/~jimmylin/cs431-2026f/corpus-enwiki-sample.txt.gz"
CORPUS_GZ_MD5 = "feb030231d6746d3a3c7c59d8c96a0bc"
CORPUS_MD5 = "4fe0c91d72b237fd938406550ad7ec60"

# The corpus goes in data/ next to this notebook. If you already have an uncompressed copy
# elsewhere, point the environment variable CS431_A5_CORPUS at it instead.
CORPUS_GZ_PATH = os.getenv("CS431_A5_CORPUS_GZ", os.path.join("data", "corpus-enwiki-sample.txt.gz"))
CORPUS_PATH = os.getenv("CS431_A5_CORPUS", os.path.join("data", "corpus-enwiki-sample.txt"))

def md5sum(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def check_md5(path, expected):
    if md5sum(path) != expected:
        raise ValueError(f"{path} is corrupt or incomplete (wrong MD5 checksum): delete it and re-run this cell.")

def write_atomically(path, stream):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path + ".part", "wb") as out:
        shutil.copyfileobj(stream, out, 1 << 20)
    os.replace(path + ".part", path)

if not os.path.exists(CORPUS_PATH):
    if not os.path.exists(CORPUS_GZ_PATH):
        print(f"Downloading {CORPUS_URL} (716 MB)...")
        with urllib.request.urlopen(CORPUS_URL) as response:
            write_atomically(CORPUS_GZ_PATH, response)
    check_md5(CORPUS_GZ_PATH, CORPUS_GZ_MD5)
    print(f"Uncompressing into {CORPUS_PATH} (1.8 GB)...")
    with gzip.open(CORPUS_GZ_PATH, "rb") as uncompressed:
        write_atomically(CORPUS_PATH, uncompressed)
check_md5(CORPUS_PATH, CORPUS_MD5)

print(f"corpus_bytes={os.path.getsize(CORPUS_PATH)}")
print("checksum=ok")

You should see `checksum=ok`.
Don't continue until you do.

The cell only downloads and uncompresses the first time; when you re-run it, it just re-verifies the checksum, which takes a few seconds.
Do not commit the `data/` directory (or any of the indexes you build) to your repository.

Here are the first three lines of the file:

<pre style="max-width: 100%; overflow-x: auto;">
0	Anarchism.0004	Anarchism is usually considered a far-left ideology and much of anarchist economics and anarchist legal philosophy reflects anti-authoritarian interpretations of communism , collectivism , syndicalism , mutualism , or participatory economics .
1	Anarchism.0027	By the time of the French Revolution , some such as the Enraged Ones began to use the term positively in opposition to Jacobin centralisation of power , seeing `` revolutionary government '' as oxymoronic .
2	Anarchism.0037	His aversion to the imposition of a rules-based society led him to denounce as a manifestation of the people 's `` mental enslavement '' the foundations of law , property rights and even the institution of marriage .
</pre>

Each line is the docid (uniquely numbered), followed by a tab, followed by the actual "document" content.
The docids run from 0 to 10,805,213 in the order the lines appear, so a document's docid is also its zero-based line number in the file.
The content is the article title, followed by a tab, followed by a sentence.
The title is the Wikipedia article name followed by the sentence's position in the article (for example, `Anarchism.0004`).
For the purposes of indexing, treat the title and the sentence together as "the document".

The cell below reads the corpus into an RDD of `(docid, text)` pairs, where `text` is everything after the docid (that is, `title<TAB>sentence`).

Just run the cell.

In [ ]:
# Each input line is "docid<TAB>text".
# Use this function to parse.
def parse_line(line):
    parts = line.split("\t", 1)
    if len(parts) == 2:
        return int(parts[0]), parts[1]
    else:
        return None

lines = sc.textFile(CORPUS_PATH)
docs = lines.map(parse_line).filter(lambda doc: doc is not None)

print(f"text_partition_n={lines.getNumPartitions()}")
docs.take(2)

Before proceeding, make sure you understand the corpus structure and what we just did above.

### 1.2 Tokenization

We provide the tokenizer to use in this assignment. 
Run the cell below.

**Query convention:** query operands are already tokenized strings.
The lookup functions lowercase them but do not run `tokenize` on them.
Thus `WATERLOO` looks up `waterloo`, while `Waterloo,` looks up `waterloo,` and usually finds nothing.
Operators are uppercase `AND` and `OR`.
Keeping this convention explicit lets us study indexing tokenization separately from query parsing.

In [ ]:
import re

# Use this function to tokenize text.
def tokenize(line):
    TOKEN_RE = re.compile(r'(^[^a-z]+|[^a-z]+$)')
    out = []
    for tok in line.split():
        w = TOKEN_RE.sub('', tok.lower())
        if w:
            out.append(w)
    return out

Let's try it out on a sample of documents:

In [ ]:
first_docs = dict(docs.take(200))

for docid in [0, 191]:
    title, sentence = first_docs[docid].split("\t", 1)
    print(f"docid={docid}")
    print("title_tokens=", tokenize(title))
    print("sentence_tokens=", tokenize(sentence))
    print()

**Q1.2(a): Tokenization examples**

How does the tokenizer tokenize the following terms? 
`["Waterloo,", "U.S.", "1990s", "C++", "2004", "don't", "e-mail", "``", "Zürich", "Anarchism.0004"]`

That is, for each of the strings above, what is the tokenized output?

// acell_512_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q1.2(b): What does the tokenizer remove and keep?**

The question above is asking about specific example strings.
Here, we want you to describe at a higher level what the tokenizer is doing.

// acell_512_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


## 2. Building an Uncompressed Inverted Index

### 2.1 Postings Lists (Uncompressed)

An inverted index maps each term to its _postings list_: the documents that contain the term.
In our index, each posting is a pair `(docid, tf)`, where `tf` (term frequency) is the number of times the term occurs among the document's tokens (title and sentence together, exactly as produced by `tokenize`).

For reference, we want you to implement the inverted indexing algorithm described on slide 25 [here](https://lintool.github.io/cs431-2026f/slides/cs431-2026f-week09b-slides-v1.00.pdf#page=25).
The pseudo-code is given in MapReduce, but we want you to implement it in Spark.

Implement `build_postings(docs)`:

+ Input: an RDD of `(docid, text)` pairs, like `docs`.
+ Output: an RDD with exactly one record `(term, postings)` per distinct term, where `postings` is a Python list of `(docid, tf)` tuples sorted by docid in ascending order.
+ Use Spark RDD transformations. Do not collect the corpus or the index in the driver.

The algorithm works as follows: count each document’s tokens, emit one `(term, (docid, tf))` record per distinct term in that document, group records by term, and sort each group by docid. This buffers all postings for a term before sorting them by docid.
Grouping can concentrate many postings in one task, and sorting materializes a term's entire list. We will analyze these costs after inspecting the index statistics.

To help you along the way, consider this (toy) unordered input:

```python
[(3, "Toy.0003\tA cat, a dog, a CAT!"),
 (0, "Toy.0000\tThe cat sat on the mat.")]
```

Selected output records are `("cat", [(0, 1), (3, 2)])`, `("a", [(3, 3)])`, and `("toy", [(0, 1), (3, 1)])`.
Note that repeated tokens increase `tf` rather than producing duplicate postings.
Docids in the input are unique non-negative integers, but input order is arbitrary; output record order across terms is unspecified.
An empty input produces no records. 
A document with no tokens contributes no postings.

In [ ]:
## acell_521_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

Let's check your function on a tiny corpus where we know the answer.
Just run the cell.

In [ ]:
TOY_DOCS = [
    (3, "Toy.0003\tA cat, a dog, a CAT!"),
    (0, "Toy.0000\tThe cat sat on the mat."),
    (2, "Toy.0002\tThe dog sat."),
    (1, "Toy.0001\t1990s: the U.S. e-mail era"),
]

TOY_POSTINGS = {
    "a": [(3, 3)], "cat": [(0, 1), (3, 2)], "dog": [(2, 1), (3, 1)], "e-mail": [(1, 1)],
    "era": [(1, 1)], "mat": [(0, 1)], "on": [(0, 1)], "s": [(1, 1)], "sat": [(0, 1), (2, 1)],
    "the": [(0, 2), (1, 1), (2, 1)], "toy": [(0, 1), (1, 1), (2, 1), (3, 1)], "u.s": [(1, 1)],
}

toy_postings = build_postings(sc.parallelize(TOY_DOCS, 3)).collect()
assert len(toy_postings) == len(TOY_POSTINGS), "expected exactly one record per term"
assert dict(toy_postings) == TOY_POSTINGS, "postings differ from the expected toy index"
print("build_postings_checks=passed")

You should see `build_postings_checks=passed`.
If not, stop and debug.

### 2.2 Writing the Index (Uncompressed)

Now let's turn the index into text, one line per term:

```text
term<TAB>df docid:tf docid:tf ...
```

That is, each line starts with the term, followed by a tab; the rest of the line is the postings list itself.
The postings list begins with the `df` (document frequency): the number of documents that contain the term.
After the `df`, there are exactly that many space-separated `docid:tf` pairs, in ascending order of docid.
Across all the part files in the index directory, there is exactly one line for each unique term in the vocabulary.

For example, a term that occurs once in document 3 and twice in document 17 becomes the line `term<TAB>2 3:1 17:2`.

Create `postings_lists`, an RDD with one string per term, in exactly this format, built from `build_postings(docs)`.
Below, we're going to take `postings_lists`, call `saveAsTextFile`, and that will be our uncompressed inverted index.

_Hints:_

+ `build_postings(docs)` already gives you one `(term, postings)` record per term, with the postings in docid order, so a single `map` is enough. In the map you just need to do some string manipulation in Python.
+ Don't call any actions (such as `collect` or `count`) yourself.

Once you complete the implementation and run the cell, you'll build the index for the full corpus by writing `postings_lists` to disk, which may take a few minutes.
(It might make sense during debugging to first try with a subset of the corpus.)

The cell below will print how long the indexing took (its "Wall time"); we'll use it in Section 5.

In [ ]:
## acell_522_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

import shutil
import time

UNCOMPRESSED_INDEX_DIR = "index-enwiki-uncompressed"

shutil.rmtree(UNCOMPRESSED_INDEX_DIR, ignore_errors=True)

build_times = {}
start = time.perf_counter()
postings_lists.saveAsTextFile(UNCOMPRESSED_INDEX_DIR)
build_times["text"] = time.perf_counter() - start
print(f"Wall time: {build_times['text']:.1f} s")

Inside the notebook, you won't get any feedback while the Spark job is running, but you can always go to the Spark job tracker to find out what's happening.
Typically, the Spark job tracker is at `http://localhost:4040/jobs/`.

Let's look at what was written: how many part files there are, the size of the index, and its first few lines (cut to 100 characters).

In [ ]:
from pathlib import Path

def apparent_bytes(path):
    path = Path(path)
    return path.stat().st_size if path.is_file() else sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

part_paths = sorted(Path(UNCOMPRESSED_INDEX_DIR).glob("part-*"))
print(f"output_partition_n={postings_lists.getNumPartitions()}")
print(f"part_file_n={len(part_paths)}")
print(f"index_apparent_bytes={apparent_bytes(UNCOMPRESSED_INDEX_DIR)}")
print()

print("First three lines of the first part file, cut to 100 characters:")
with part_paths[0].open(encoding="utf-8") as f:
    for _, line in zip(range(3), f):
        print(line.rstrip("\n")[:100] + "...")

The `saveAsTextFile` action writes one part file per output partition, with each task writing its own file. 
The displayed `output_partition_n` and `part_file_n` should therefore agree.

Let's check the format of the first lines written.
Just run the cell.

In [ ]:
# Each line: term, tab, df, then exactly df docid:tf pairs.
LINE_RE = re.compile(r"^\S+\t(\d+)((?: \d+:\d+)+)$")
for line in sc.textFile(UNCOMPRESSED_INDEX_DIR).take(1000):
    match = LINE_RE.match(line)
    assert match and int(match.group(1)) == match.group(2).count(":"), f"badly formatted line: {line[:80]!r}"
print("format_checks=passed")

You should see `format_checks=passed`. If not, stop and debug.

Next, let's probe to determine some statistics about the index you just wrote: the number of terms, the total number of postings (the sum of all the `df`s), the five terms with the largest `df`, and the `df` of the token `s`.

Just run the cell.

In [ ]:
# (term, df) for every line of the index: "term<TAB>df docid:tf ..." -> ("term", df)
term_dfs = (
    sc.textFile(UNCOMPRESSED_INDEX_DIR)
    .map(lambda line: line.split(" ", 1)[0].split("\t"))
    .map(lambda pair: (pair[0], int(pair[1])))
)

term_n = term_dfs.count()
print(f"term_n={term_n}")
print(f"posting_n={term_dfs.values().sum()}")
print()

print("The 5 terms with the largest df:")
for term, df in term_dfs.takeOrdered(5, key=lambda pair: -pair[1]):
    print(f"{term:>6} {df}")
print()

print(f"df of \"s\"={dict(term_dfs.filter(lambda pair: pair[0] == "s").collect()).get("s", 0)}")

**Q2.2(a): "Doubling the number of workers halves the memory needed to build the largest postings list."**

Is this statement true for `build_postings`? If so, explain why; if not, justify your answer. Consider:

1. How `groupByKey` distributes postings for a single term.
2. What a task must hold in memory to sort that term's postings.

Use the largest `df` displayed above to support your answer.

// acell_522_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


## 3. Boolean Retrieval on an Uncompressed Inverted Index

With the index on disk, we can answer queries (i.e., perform Boolean retrieval).
Our search engine runs in plain Python in this notebook's process (the driver), not in Spark.
It loads the index into memory and evaluates each query a term at a time: it looks up the postings list for each query term and combines it with the result so far.

### 3.1 Loading the Index and the Documents

Before we can search, the search engine first loads into memory the index you wrote in Section 2.2.

Run the following cell to create a dictionary we're going to use to keep track of loading times:

In [ ]:
load_times = {}

Implement `load_uncompressed_index(index_path)`:

+ `index_path` is either a directory of part files, like the one `saveAsTextFile` wrote in Section 2.2, or a single index file.
+ In a directory, read every line of every `part-*` file, and nothing else: Spark also writes other files there, such as `_SUCCESS` and hidden checksum files (e.g., `.part-00000.crc`).
+ Return a dictionary that maps each term to its line of the index, _as text_: do not parse the postings. (A trailing newline is fine.)

We've given you the docstring, the timing code, and comments for the two steps; fill in the steps.

For example, a directory containing these files:

```text
part-00000: cat<TAB>2 0:1 3:2
part-00001: dog<TAB>1 3:1
_SUCCESS:  (empty)
.part-00000.crc: (checksum metadata)
```

should produce `{"cat": "cat\t2 0:1 3:2\n", "dog": "dog\t1 3:1\n"}`. Only regular files whose names begin with `part-` are index data. A directory with no such files returns `{}`. Input index records are valid and terms are unique across files.

In [ ]:
## acell_531_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.
# Fill in the two steps between the markers, inside the function.

def load_uncompressed_index(index_path: str):
    """
    Load uncompressed text index into dict: term -> index line
    - Works if index_path is a single file OR a directory of part files.
    """
    import time
    from datetime import datetime, timezone

    start = time.perf_counter()
    wall_start = time.time()
    start_str = datetime.fromtimestamp(wall_start, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print(f'Loading index...')
    print(f'Start time: {start_str}')

    index = {}
    paths = []

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

    elapsed = time.perf_counter() - start
    end = time.time()
    end_str = datetime.fromtimestamp(end, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

    print(f'End time: {end_str}')
    print(f'Total elapsed time: {elapsed:.3f}s')
    print()

    load_times["text"] = elapsed

    return index

Note that the cell above just defines `load_uncompressed_index`.
It doesn't run anything.

Let's check your function on the whole index directory (`UNCOMPRESSED_INDEX_DIR`, which is `index-enwiki-uncompressed`).

Just run the cell.
It may take a little while to load everything into memory.

In [ ]:
import contextlib
import io

# One part file first: a quick check before loading everything
first_part_path = os.path.join(UNCOMPRESSED_INDEX_DIR, "part-00000")
with contextlib.redirect_stdout(io.StringIO()):  # hide the timing output for this check
    part_index = load_uncompressed_index(first_part_path)
with open(first_part_path, encoding="utf-8") as f:
    assert len(part_index) == sum(1 for _ in f), f"expected one entry per line of {first_part_path}"
assert all(type(line) is str for line in part_index.values()), "each value must be the term's line, as text (not parsed)"

# The whole index directory
uncompressed_index = load_uncompressed_index(UNCOMPRESSED_INDEX_DIR)

assert len(uncompressed_index) == term_n, f"expected {term_n} terms (see Section 2.2), got {len(uncompressed_index)}: are you reading only the part-* files?"
assert all(type(line) is str and line.startswith(term + "\t") for term, line in uncompressed_index.items()), "each value must be the term's line, as text"
assert all(uncompressed_index.get(term) == line for term, line in part_index.items())
print(f"loaded_term_n={len(uncompressed_index)}")
print("load_checks=passed")

del part_index  # Avoid retaining an unnecessary copy of one partition.

Each value in `uncompressed_index` is now a line of the index, exactly as you wrote it in Section 2.2.
You should see `load_checks=passed`.
If not, stop and debug.

To search, we need to turn a line back into a postings list.

Implement `parse_index_line(line)`:

+ `line` is one line of the index, in the format from Section 2.2: `term<TAB>df docid:tf docid:tf ...`. It may end with a newline, as lines read from a file do.
+ Return a tuple `(term, postings)`, where `postings` is a list of `(docid, tf)` tuples of integers sorted by docid in ascending order.

For example, `parse_index_line("cat\t2 1:2 5:1\n")` returns `("cat", [(1, 2), (5, 1)])`.

Inputs are guaranteed to be valid records with non-negative docids, positive term frequencies, and a `df` matching the number of postings. You do not need to repair malformed records. A valid `df=0` record returns an empty postings list. The returned postings must be sorted even if the input pairs are not.

In [ ]:
## acell_531_2 (keep this id for tracking purposes)
# TODO: Replace only the implementation between the markers; keep the signature.

######### BEGIN STUDENT CODE #########

def parse_index_line(line: str):
    # Code here...
    pass

######### END STUDENT CODE #########

Let's run some checks.
The cell also defines `uncompressed_lookup(term)`, which uses your `parse_index_line` to return the set of docids for a term, or an empty set if the term isn't in the index.

Just run the cell.

In [ ]:
from itertools import islice

assert parse_index_line("cat\t2 1:2 5:1\n") == ("cat", [(1, 2), (5, 1)])
assert parse_index_line("u.s\t1 10805213:3") == ("u.s", [(10805213, 3)])

# Round trip: the toy postings, written in the format of Section 2.2, then parsed back
for term, postings in TOY_POSTINGS.items():
    line = "{}\t{} {}\n".format(term, len(postings), " ".join(f"{d}:{tf}" for d, tf in postings))
    assert parse_index_line(line) == (term, postings), f"wrong result for {line!r}"

# The first 1000 lines of the real index
for term, line in islice(uncompressed_index.items(), 1000):
    parsed_term, postings = parse_index_line(line)
    assert parsed_term == term
    assert len(postings) == int(line.split("\t", 1)[1].split(" ", 1)[0]), f"df mismatch for {term!r}"
    assert all(type(d) is int and type(tf) is int for d, tf in postings)
    assert all(a[0] < b[0] for a, b in zip(postings, postings[1:])), f"postings not sorted for {term!r}"
print("parse_checks=passed")

def uncompressed_lookup(term):
    line = uncompressed_index.get(term.lower())
    if not line:
        return set()
    _, lst = parse_index_line(line)
    return {d for d, _ in lst}

print(f"waterloo_postings_head={sorted(uncompressed_lookup('waterloo'))[:5]}")

assert parse_index_line("cat\t0\n") == ("cat", [])
assert parse_index_line("cat\t2 3:2 0:1") == ("cat", [(0, 1), (3, 2)])
assert uncompressed_lookup("WATERLOO") == uncompressed_lookup("waterloo")
assert uncompressed_lookup("1990s") == set()

You should see `parse_checks=passed`.
If not, stop and debug.

**Q3.1(a): Why parse postings only when a term is looked up?**

Look at `uncompressed_lookup("waterloo")` above. 
Which postings list does it parse, and why would parsing every list when loading the index do unnecessary work for this lookup?

// acell_531_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


To print the results, we also need each document's title and sentence.
The cell below loads the corpus into a dictionary from docid to text, using `parse_line` from Section 1.1.

Just run the cell. 
It may take a little while since we're loading the entire corpus into memory.

In [ ]:
# Load documents as docid-text pairs
def load_collection(path: str):
    import time
    from datetime import datetime, timezone

    start = time.time()
    start_str = datetime.fromtimestamp(start, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print(f'Loading corpus...')
    print(f'Start time: {start_str}')

    lines_by_id = {}
    with open(path, "r", encoding="utf-8") as fin:
        for line in fin:
            parsed = parse_line(line)
            if parsed:
                d, text = parsed
                lines_by_id[d] = text

    end = time.time()
    end_str = datetime.fromtimestamp(end, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

    print(f'End time: {end_str}')
    print(f'Total elapsed time: {end - start:.0f}s')
    print()

    return lines_by_id

lines_by_id = load_collection(CORPUS_PATH)

print(f"loaded_doc_n={len(lines_by_id)}")

### 3.2 Evaluating Queries

Our query language is deliberately simple: terms separated by the operators `AND` and `OR`, for example `waterloo OR stanford AND cheriton`.
Instead of supporting arbitrarily nested Boolean expressions, we make a significant simplification: queries are interpreted strictly from left to right.
That is, `a OR b AND c` is interpreted as `(a OR b) AND c`: read the query from left to right, and apply each operator to the result so far and the next term.

Implement `evaluate_query(query_str, term_docs)`:

+ `query_str` is a string; split it on whitespace. Expect: term, operator, term, operator, ..., term.
+ `term_docs(term)` returns the set of docids for a term (an empty set if the term isn't in the index).
+ Start with the first term's docids, then walk left to right: `AND` intersects the result so far with the next term's docids, and `OR` takes their union.
+ If a token where an operator is expected is neither `AND` nor `OR`, stop early and return the result so far.
+ Return the resulting set of docids (an empty set for an empty query).

For `cat OR dog AND sat`, suppose `cat -> {0, 3}`, `dog -> {2, 3}`, and `sat -> {0, 2}`. The intermediate result is `{0, 2, 3}` after `OR`, then `{0, 2}` after `AND`.

Operands are passed unchanged to `term_docs`; the supplied lookup functions perform lowercasing only. Operators are case-sensitive. An unsupported operator stops evaluation, and a trailing operator with no operand is ignored: both `cat XOR dog` and `cat AND` return the docids for `cat`. Do not add parentheses, `NOT`, or conventional operator precedence. Do not mutate sets returned by `term_docs`.

In [ ]:
## acell_532_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

If the above description isn't clear, see below for the desired behavior.

Let's run some checks on a toy index.
Just run the cell.

In [ ]:
TOY_INDEX = {"cat": {0, 3}, "dog": {2, 3}, "the": {0, 1, 2}, "sat": {0, 2}}
toy_term_docs = lambda term: TOY_INDEX.get(term.lower(), set())

assert evaluate_query("cat", toy_term_docs) == {0, 3}
assert evaluate_query("cat AND dog", toy_term_docs) == {3}
assert evaluate_query("cat OR dog", toy_term_docs) == {0, 2, 3}
assert evaluate_query("cat OR dog AND sat", toy_term_docs) == {0, 2}
assert evaluate_query("dog AND sat OR cat", toy_term_docs) == {0, 2, 3}
assert evaluate_query("Cat AND fish", toy_term_docs) == set()
assert evaluate_query("cat XOR dog", toy_term_docs) == {0, 3}
assert evaluate_query("", toy_term_docs) == set()
print("query_checks=passed")

assert evaluate_query("cat AND", toy_term_docs) == {0, 3}
assert evaluate_query("cat and dog", toy_term_docs) == {0, 3}
assert TOY_INDEX["cat"] == {0, 3}, "do not mutate the lookup's sets"

You should see `query_checks=passed`.
If not, stop and debug.

### 3.3 Running Queries

Let's put everything together.
The cell below defines `print_results`, which prints the matching documents in ascending order of docid: the docid, the Wikipedia article title, and the sentence.
(The title and sentence are trimmed.)

It then runs a few queries against the uncompressed index, printing the results of each and recording result sets.
Repeated query timings are measured together in Section 5.2 so the two representations use the same procedure.

Just run the cell.

In [ ]:
def print_results(current):
    for d in sorted(current):
        line = lines_by_id.get(d, '').replace('\n', '').replace('\r', '')
        parts = line.split("\t", 1)
        docid = d
        title = parts[0]
        sentence = parts[1]
        sentence_trimmed = sentence[:80] + "..." if len(sentence) > 80 else sentence
        title_trimmed = title[:20] + "..." if len(title) > 20 else title        
        print(f'{docid:8} {title_trimmed:<23} {sentence_trimmed}')

QUERIES = [
    "waterloo AND canada AND ontario",
    "waterloo AND belgium",
    "waterloo OR stanford AND cheriton",
    "big AND data AND hadoop",
    "outrageous AND fortune",
    "university AND of AND waterloo",
]

uncompressed_results = {}
for query in QUERIES:
    uncompressed_results[query] = evaluate_query(query, uncompressed_lookup)
    print(f"query={query!r} hits={len(uncompressed_results[query])}")
    print_results(uncompressed_results[query])
    print()

Check a few returned documents by hand, remembering the evaluation order for mixed operators.

For the next question, code is provided.
It displays one matching sentence from the article *University of Sydney* in full (rather than a truncated result).
The token positions below refer to the combined title and sentence.

In [ ]:
phrase_terms = ["university", "of", "waterloo"]
sydney_matches = [d for d in sorted(uncompressed_results["university AND of AND waterloo"])
                  if lines_by_id[d].split("\t", 1)[0].startswith("University of Sydney.")]
assert sydney_matches, "expected the Sydney example in the full corpus"

sydney_docid = sydney_matches[0]
sydney_title, sydney_sentence = lines_by_id[sydney_docid].rstrip("\r\n").split("\t", 1)
print(f"docid={sydney_docid} title={sydney_title}")
print(sydney_sentence)
print()

sydney_tokens = tokenize(lines_by_id[sydney_docid])
for term in phrase_terms:
    print(term, "token_positions=", [i for i, token in enumerate(sydney_tokens) if token == term])

**Q3.3(a): Why does the Sydney sentence match?**

The query is `university AND of AND waterloo`.
Why are we retrieving a sentence about the University of Sydney?

// acell_533_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


The next toy example connects the lexical-overlap limitation to the sparse/dense discussion in the slides.

In [ ]:
semantic_passage = "A prestigious post-secondary institution in Ontario with a strong emphasis on STEM."
semantic_query = "university of waterloo"
passage_terms, query_terms = set(tokenize(semantic_passage)), set(tokenize(semantic_query))
print("passage=", semantic_passage)
print("query=", semantic_query)
print("passage_tokens=", sorted(passage_terms))
print("query_tokens=", sorted(query_terms))
print("overlap=", sorted(passage_terms & query_terms))
print("multi_hot_dot_product=", len(passage_terms & query_terms))

**Q3.3(b): What can lexical matching miss?**

Why is the displayed dot product zero despite the passage being relevant to the query? Briefly explain how dense representations could help.

// acell_533_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


## 4. Compressing the Index

Compare the corpus size printed during setup with the index size in Section 2.2.
You'll see that the index size can exceed the corpus size because terms map to many explicit docid/tf pairs (represented as strings).

In this section, we'll represent each postings list in binary, using variable-byte (VByte) codes for all integers and gap (delta) encoding for docids.
We explain both encodings below.
We will also measure an index variant with VByte but no gaps.

### 4.1 Variable-Byte Codes

We explained variable-byte codes in class.

Below, we provide the implementation of `vint_encode(n)` to encode a non-negative integer as a `bytes` object.
Each byte has seven payload bits and one continuation bit:

```text
bit:     7 | 6 5 4 3 2 1 0
         ^   seven payload bits
         continuation flag
```

We use as many bytes as needed.
The implementation writes the least significant seven-bit group first. 
The high bit is set to 1 if more groups follow, or 0 for the final byte.
Even zero must produce one byte, `bytes([0])`.

Just run the cell below.

In [ ]:
def vint_encode(n):
    # variable int encoding into a byte array.
    # We’ll use a standard 7-bit continuation encoding, which we will also decode on the reader side.
    b = bytearray()
    while True:
        to_write = n & 0x7F
        n >>= 7
        if n:
            b.append(to_write | 0x80)
        else:
            b.append(to_write)
            break

    return bytes(b)

**Q4.1(a): How are various integers encoded?**

For each of the following integers, provide the encoding: `[0, 5, 127, 128, 300, 16_383, 16_384, 10_805_213]`.
Specifically:

+ How many bytes are required?
+ What are the actual bits?
  + If one byte, the answer should be in the form of `xxxxxxxx`, where `x` is either 0 or 1.
  + If two bytes, the answer should be in the form of `xxxxxxxx` `xxxxxxxx`, where `x` is either 0 or 1.
  + And so on for more bytes...

// acell_541_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


Now we give you the implementation of the decoder, `vint_decode_one(buf, i)`:

+ `buf` is a `bytes` object (or a `memoryview` of one) containing one or more concatenated `vint_encode` outputs; `i` is the index of the first byte of a number.
+ The decoder returns a tuple `(value, next_index)`: the decoded integer, and the index of the byte just after it.
+ The decoder exactly inverts `vint_encode`, for integers of any size.

The payload groups are stored least significant first.
A high bit of 1 means another byte follows; 0 terminates the current integer. 
Inputs are valid, complete encodings with `i` at a number's first byte.
Malformed or truncated buffers are outside the required contract.

For example, in `bytes([5, 172, 2, 7])`, decoding at `i=1` returns `(300, 3)`. 
The `3` is an absolute index in the original buffer, not the two-byte length of the decoded number.
A caller can use that index to decode the next integer.

Just run the following cell.

In [ ]:
def vint_decode_one(buf, i):
    # returns (value, next_index)
    shift = 0
    n = 0
    while True:
        b = buf[i]
        i += 1
        n |= (b & 0x7F) << shift
        if (b & 0x80) == 0:
            break
        shift += 7
    return n, i

Let's run some checks. Just run the following cell.

In [ ]:
assert vint_decode_one(bytes([0b10101100, 0b00000010]), 0) == (300, 2)
assert vint_decode_one(bytes([5, 0b10101100, 0b00000010, 7]), 1) == (300, 3)

values = [0, 1, 127, 128, 255, 300, 16_383, 16_384, 2**21, 10_805_213, 2**35 + 7]
stream = b"".join(vint_encode(v) for v in values)
pos, decoded = 0, []
while pos < len(stream):
    value, pos = vint_decode_one(stream, pos)
    decoded.append(value)
assert decoded == values
print("vint_checks=passed")

Make sure you understand how the encoder and decoder work.

### 4.2 Gap Encoding

Since a postings list is sorted by docid, we can store the _difference_ (gap) between consecutive docids instead of the docids themselves.
Gaps are usually much smaller than docids, and VByte stores small numbers in fewer bytes.

Implement `encode_postings(postings)` and `decode_postings(bts)`:

+ `postings` is a list of `(docid, tf)` tuples sorted by docid ascending (possibly empty).
+ `encode_postings` returns a `bytes` object: `vint_encode(df)`, followed by, for each posting in order, `vint_encode(gap)` then `vint_encode(tf)`, where `gap` is the docid minus the previous posting's docid. For the first posting, the gap is the docid itself.
+ `decode_postings` exactly inverts `encode_postings`, returning a list of `(docid, tf)` tuples.

Use `vint_encode` and `vint_decode_one` provided above.

Docids are distinct non-negative integers and `tf` values are positive integers. Input byte strings are valid, complete encodings produced by this format; handling malformed buffers is not required.

For example:

```text
postings:       [(0, 2), (3, 1), (300, 5)]
integer stream: [3,       0, 2,    3, 1,    297, 5]
                 df      gap tf   gap tf   gap  tf
encoded bytes:  [3, 0, 2, 3, 1, 169, 2, 5]
```

Decode `df` first, then exactly that many gap/tf pairs, cumulatively adding gaps to reconstruct docids. For the empty list, encode only `df=0`, yielding `bytes([0])`. The first gap can be zero because docid zero is valid.

In [ ]:
## acell_542_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

Let's run some checks. Just run the cell.

In [ ]:
example = [(3, 1), (7, 2), (300, 1), (301, 5)]
assert encode_postings(example) == bytes([4, 3, 1, 4, 2, 0b10100101, 0b00000010, 1, 1, 5])
assert decode_postings(encode_postings(example)) == example
assert encode_postings([]) == bytes([0]) and decode_postings(bytes([0])) == []

waterloo = parse_index_line(uncompressed_index["waterloo"])[1]
assert decode_postings(encode_postings(waterloo)) == waterloo
print("gap_checks=passed")

zero_example = [(0, 2), (3, 1), (300, 5)]
assert encode_postings(zero_example) == bytes([3, 0, 2, 3, 1, 169, 2, 5])
assert decode_postings(bytes([3, 0, 2, 3, 1, 169, 2, 5])) == zero_example

For selected terms, compare the bytes per posting of text, VByte with absolute docids, and VByte with gaps.
These values include each list's `df` header, but exclude the term string and outer file-format overhead.
The text value also excludes its final newline.

The mean gap and percentage of gaps below 128 help explain the encoded sizes: a gap below 128 fits in one VByte byte.
Note that frequency alone does not determine gap sizes: documents containing a term can cluster, including sentences from one article.

In [ ]:
def encode_postings_without_gaps(postings):
    out = bytearray(vint_encode(len(postings)))
    for docid, tf in postings:
        out += vint_encode(docid) + vint_encode(tf)
    return bytes(out)

print(f"{'term':>12} {'df':>9} {'text':>7} {'vbyte':>7} {'gaps':>7} {'mean_gap':>12} {'gap<128 (%)':>12}")
for term in ["the", "university", "waterloo", "hadoop", "zipf"]:
    line = uncompressed_index[term]
    _, postings = parse_index_line(line)
    df = len(postings)
    text_bytes = len(line.rstrip("\n").split("\t", 1)[1].encode("utf-8"))
    no_gap_bytes = len(encode_postings_without_gaps(postings))
    gap_bytes = len(encode_postings(postings))
    previous = 0
    gap_sum = small_gap_n = 0
    for docid, _ in postings:
        gap = docid - previous
        gap_sum += gap
        small_gap_n += gap < 128
        previous = docid
    print(f"{term:>12} {df:>9} {text_bytes/df:>7.2f} {no_gap_bytes/df:>7.2f} {gap_bytes/df:>7.2f} {gap_sum/df:>12.1f} {100*small_gap_n/df:>12.2f}")
    del postings
print("The text, vbyte, and gaps columns are bytes per posting, including the df header.")

**Q4.2(a): Does document frequency determine compressed size?**

Suppose two terms have the same `df` and every posting has `tf=1`. Must their gap-encoded lists occupy the same number of bytes? Explain.

// acell_542_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


### 4.3 Building the Compressed Index

Now we can build the compressed index: one record per term, whose value is the term's postings list encoded with `encode_postings`.
From a logical perspective, the postings lists are identical to the uncompressed ones (the same `df` and `(docid, tf)` pairs); only the physical representation (i.e., encoding) changes.

The implementation of `build_compressed_index(docs)` is provided for you below:

+ Input: an RDD of `(docid, text)` pairs, like `docs`.
+ Output: an RDD with exactly one record `(term, encoded_postings)` per distinct term, where `encoded_postings` is `encode_postings` applied to the term's postings list (sorted by docid, exactly as in Section 2).

The implementation uses Spark RDD transformations.
Note that it does _not_ collect the corpus or the index in the driver.
It returns an RDD without invoking an action. 
For example, the selected toy term `cat` has postings `[(0, 1), (3, 2)]`, so its value is `bytes([2, 0, 1, 3, 2])`.
We reuse earlier code.

In [ ]:
def build_compressed_index(docs):
    return build_postings(docs).mapValues(encode_postings)

Let's check your function on the toy corpus. Just run the cell.

In [ ]:
toy_compressed = build_compressed_index(sc.parallelize(TOY_DOCS, 3)).collect()
assert len(toy_compressed) == len(TOY_POSTINGS), "expected exactly one record per term"
assert dict(toy_compressed) == {term: encode_postings(p) for term, p in TOY_POSTINGS.items()}
print("compressed_build_checks=passed")

You should see `compressed_build_checks=passed`.
If not, stop and debug.

Now build the compressed index for the full corpus.
The cell below runs your `build_compressed_index`, brings the result back to the driver, and saves it as a single file (a pickled dictionary from term to encoded postings).

Just run the cell, which may take a few minutes.

In [ ]:
import pickle
import time

COMPRESSED_INDEX_PATH = "index-enwiki-compressed.pkl"

start = time.perf_counter()

# Bring the index to the driver and dump it
term_bytes = dict(build_compressed_index(docs).collect())          # {term: bytes}
payload = {"index": term_bytes}

with open(COMPRESSED_INDEX_PATH, "wb") as f:
    pickle.dump(payload, f)

print(f"Wrote pickle index with {len(term_bytes):,} terms to {COMPRESSED_INDEX_PATH}")
build_times["vbyte_gaps"] = time.perf_counter() - start
print(f"Wall time: {build_times['vbyte_gaps']:.1f} s")
del term_bytes, payload

import resource

print(f"compressed_file_bytes={apparent_bytes(COMPRESSED_INDEX_PATH)}")
print(f"spark_driver_memory={sc.getConf().get('spark.driver.memory')}")
print(f"spark_driver_maxResultSize={sc.getConf().get('spark.driver.maxResultSize')}")

# On Linux ru_maxrss is KiB; this is a high-water mark for this Python process only.
if sys.platform.startswith("linux"):
    print(f"python_peak_rss_gib={resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.2f}")

print("File size, serialized Spark results, and peak live Python memory are different quantities.")

**Q4.3(a): Why does collecting the index limit scalability?**

Explain why the `collect` step could fail on a much larger corpus and how the index could be written without gathering it on the driver.

// acell_543_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


### 4.4 Full-Index Baseline: VByte Without Gaps

Let's examine the effects of gap compression.
Build another index with absolute docids and VByte for every integer, but no gap encoding.
This one-off baseline uses the same pickle wrapper and protocol as the gap-encoded index.

The provided cell reuses the already written text postings to avoid repeating tokenization and grouping.
Its elapsed time is labeled separately: its build path differs, so do not compare it as a controlled indexing-time experiment.
All three artifact sizes will appear together in Section 5.2.
(The search engine on compressed indexes continues to use the gap-encoded index.)

Just run the cell below.

In [ ]:
NO_GAP_INDEX_PATH = "index-enwiki-vbyte-no-gaps.pkl"

def no_gap_record(line):
    term, postings = parse_index_line(line)
    return term, encode_postings_without_gaps(postings)

start = time.perf_counter()
no_gap_terms = dict(sc.textFile(UNCOMPRESSED_INDEX_DIR).map(no_gap_record).collect())
assert len(no_gap_terms) == term_n
with open(NO_GAP_INDEX_PATH, "wb") as f:
    pickle.dump({"index": no_gap_terms}, f)
del no_gap_terms

print(f"no_gap_file_bytes={apparent_bytes(NO_GAP_INDEX_PATH)}")
print(f"no_gap_reencode_wall_s={time.perf_counter() - start:.3f}")

## 5. Retrieval with the Compressed Index

### 5.1 Loading the Compressed Index

Implement `load_compressed_index(path)`, the counterpart of your `load_uncompressed_index`:

+ `path` is the pickle file written in Section 4.3: a pickled dictionary `{"index": {term: encoded_postings}}`.
+ Return a dictionary that maps each term to its encoded postings (`bytes`), as stored: do not decode the postings.

We've given you the docstring, the timing code, and comments for the two steps; fill in the steps.

For example, a stored object `{"index": {"cat": bytes([2, 0, 1, 3, 2])}}` produces the return value `{"cat": bytes([2, 0, 1, 3, 2])}`.
Note that the outer `"index"` key is the structure of the file format, not a term.

In [ ]:
## acell_551_1 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.
# Fill in the two steps between the markers, inside the function.

def load_compressed_index(path):
    """
    Loads a pickle inverted index written as:
      term: string
      postings: binary (vbyte+gap encoded)
    Returns: dict[str, bytes]
    """
    import time
    from datetime import datetime, timezone

    start = time.perf_counter()
    wall_start = time.time()
    start_str = datetime.fromtimestamp(wall_start, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    print(f'Loading index...')
    print(f'Start time: {start_str}')

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

    elapsed = time.perf_counter() - start
    end = time.time()
    end_str = datetime.fromtimestamp(end, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

    print(f'End time: {end_str}')
    print(f'Total elapsed time: {elapsed:.3f}s')
    print()

    load_times["vbyte_gaps"] = elapsed

    return index

Implement `compressed_lookup(term)` using the global `compressed_index` dictionary loaded below:

+ Lowercase the term and retrieve its encoded bytes; return an empty set for an absent term.
+ Decode only that term's postings with `decode_postings`, then return its docids as a set.
+ Keep stored postings encoded and do not mutate the dictionary.

For example, if `compressed_index["cat"] == bytes([2, 0, 1, 3, 2])`, both `compressed_lookup("cat")` and `compressed_lookup("CAT")` return `{0, 3}`. Punctuation is not stripped, following the query convention in Section 1.2.

In [ ]:
## acell_551_2 (keep this id for tracking purposes)
# TODO: Replace only the implementation between the markers; keep the signature.

######### BEGIN STUDENT CODE #########

def compressed_lookup(term: str) -> set[int]:
    # Code here...
    pass

######### END STUDENT CODE #########

Note that the two cells above just define the functions.

In the cell below, we actually load the compressed index and perform a few sanity checks.

In [ ]:
compressed_index = load_compressed_index(COMPRESSED_INDEX_PATH)
assert compressed_index.keys() == uncompressed_index.keys()
assert all(isinstance(bts, bytes) for bts in compressed_index.values())

for term in ["waterloo", "WATERLOO", "the", "zipf", "1990s", "__missing_term__"]:
    assert compressed_lookup(term) == uncompressed_lookup(term)

# Compare complete (docid, tf) pairs, rather than docids alone, for selected terms.
for term in ["waterloo", "zipf", "hadoop"]:
    assert decode_postings(compressed_index[term]) == parse_index_line(uncompressed_index[term])[1]

print(f"loaded_term_n={len(compressed_index)}")
print("compressed_load_checks=passed")

You should see `compressed_load_checks=passed`.
If not, stop and debug.

And now we can evaluate Boolean queries on the _compressed_ index.

Just run the cell below, but it's important for you to understand what's changed between the uncompressed and the compressed implementations.

In [ ]:
QUERIES = [
    "waterloo AND canada AND ontario",
    "waterloo AND belgium",
    "waterloo OR stanford AND cheriton",
    "big AND data AND hadoop",
    "outrageous AND fortune",
    "university AND of AND waterloo",
]

compressed_results = {}
for query in QUERIES:
    compressed_results[query] = evaluate_query(query, compressed_lookup)
    print(f"query={query!r} hits={len(compressed_results[query])}")
    print_results(compressed_results[query])
    print()

You should get the same results.

### 5.2 Comparing the Three Indexes

Compare all three indexes using apparent file bytes: the sum of regular file lengths for a directory or the file length for a pickle. 

The table below also consolidates the end-to-end indexing and loading times for the two search engines.
(Note that for the compressed variant _without_ gap encoding, we only perform the size analysis.)

In [ ]:
import pandas as pd

size_comparison = pd.DataFrame([
    {"variant": "text", "bytes": apparent_bytes(UNCOMPRESSED_INDEX_DIR)},
    {"variant": "vbyte_no_gaps", "bytes": apparent_bytes(NO_GAP_INDEX_PATH)},
    {"variant": "vbyte_gaps", "bytes": apparent_bytes(COMPRESSED_INDEX_PATH)},
])
size_comparison["MiB"] = size_comparison["bytes"] / 2**20
size_comparison["text_size_divided_by_variant"] = size_comparison.iloc[0]["bytes"] / size_comparison["bytes"]
print(size_comparison.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
print()
build_load_comparison = pd.DataFrame([
    {"variant": key, "build_s": build_times[key], "load_s": load_times[key]}
    for key in ["text", "vbyte_gaps"]
])
print(build_load_comparison.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

Now let's compare results for every query and measure both search engines with the same procedure: one warm-up and three measured runs, taking the median per query. 

+ Time includes lookup, parsing or decoding, set construction, and Boolean operations.
+ Time excludes index loading, corpus loading, and result printing.

Just run the cell below.

The two sets of runs still occur sequentially in one process, so CPU load, allocation, and cache effects can influence results.
Treat small differences cautiously.
The cell also checks the shared result formatting for one query and prints its compressed results.

In [ ]:
import statistics
import contextlib
import io

def benchmark_query(query, lookup, repeats=3):
    expected = evaluate_query(query, lookup)  # Warm-up; its time is not reported.
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        actual = evaluate_query(query, lookup)
        samples.append(1000 * (time.perf_counter() - start))
        assert actual == expected
    return expected, statistics.median(samples)

compressed_results = {}
uncompressed_query_ms, compressed_query_ms = {}, {}
for query in QUERIES:
    result, uncompressed_query_ms[query] = benchmark_query(query, uncompressed_lookup)
    assert result == uncompressed_results[query]
    compressed_results[query], compressed_query_ms[query] = benchmark_query(query, compressed_lookup)

mismatched_query_n = sum(compressed_results[q] != uncompressed_results[q] for q in QUERIES)
assert mismatched_query_n == 0
print(f"mismatched_query_n={mismatched_query_n}")
query_timing = pd.DataFrame({
    "query": QUERIES,
    "hits": [len(uncompressed_results[q]) for q in QUERIES],
    "uncompressed_median_ms": [uncompressed_query_ms[q] for q in QUERIES],
    "compressed_median_ms": [compressed_query_ms[q] for q in QUERIES],
})
print(query_timing.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
print("Query-term document frequencies for the large-list example:")
for term in ["university", "of", "waterloo"]:
    print(term, "df=", int(uncompressed_index[term].split("\t", 1)[1].split()[0]))

display_query = "waterloo AND belgium"
plain_output, binary_output = io.StringIO(), io.StringIO()
with contextlib.redirect_stdout(plain_output):
    print_results(uncompressed_results[display_query])
with contextlib.redirect_stdout(binary_output):
    print_results(compressed_results[display_query])
assert plain_output.getvalue() == binary_output.getvalue()
assert len(binary_output.getvalue().splitlines()) == len(compressed_results[display_query])
print("formatted_output_checks=passed")
print("Compressed results:", display_query)
print(binary_output.getvalue(), end="")

`mismatched_query_n` must be zero, and the formatting check must pass.


**Q5.2(a): Which index results in faster queries?**

Overall?
Which queries?

As noted above, be cautious of small differences.
Answer based on _both_ your empirical results above and your understanding of the implementations.

// acell_552_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.


**Q5.2(b): What makes a query slow?**

The query `university AND of AND waterloo` is relatively slow, despite returning only 305 documents. 
Why?
Explain.

Also, explain any observed difference between search latencies over the uncompressed vs. compressed indexes.

// acell_552_b (keep this id for tracking purposes)

**Answer:** Replace with your answer.


All done with Spark! Stop the session.

In [ ]:
spark.stop()

## 6. Reflection on AI Collaboration

Complete this log honestly. This is primarily for your own benefit; we'll read it, but it won't be graded.

+ What I asked the coding agent to do.
+ What worked and what needed correction.
+ One function I wrote (or rewrote) myself, and how I convinced myself it was correct beyond the provided checks.
+ One place where my lecture knowledge mattered.
+ One prompt I would write differently next time.

// acell_560_a (keep this id for tracking purposes)

**Answer:** Replace with your answer.

## 7. Assignment Submission

Before submitting:

+ Restart the kernel and run all cells from top to bottom.
+ Keep every `acell_...` identifier and every instructor-provided output statement unchanged.
+ Keep every required function, dictionary, and variable with its specified name, signature, and format.
+ Remove temporary debugging output and scratch cells, but retain the required outputs and written answers.
+ Do not commit the `data/` directory or the index files (`index-enwiki-uncompressed/`, `index-enwiki-compressed.pkl`, `index-enwiki-vbyte-no-gaps.pkl`) to your repository.
+ No standalone Python scripts or intermediate result files are required.
+ Make sure your notebook contains execution traces so that we can evaluate your implementation, for example, the sizes of the indexes, query timings, outputs supporting your written answers, etc.

For submission details, see [the course homepage](https://lintool.github.io/cs431-2026f/assignments/assignment5.html).
Make sure you conform to the requirements articulated there.